# IndicOCR on answer sheets (Colab GPU)

Runs Bodhan's open IndicOCR weights on cleaned page images and saves one JSON per page. Copy the JSON folder back and grade with `AUTOASSESS_OCR_PROVIDER=indic-precomputed`.

1. Runtime > Change runtime type > T4 GPU.
2. Accept the Indic Open Model License at https://huggingface.co/bodhan-ai/indic-ocr.
3. Create a Hugging Face read token.

In [ ]:
from huggingface_hub import login
login()  # paste your Hugging Face read token

In [ ]:
from huggingface_hub import snapshot_download
repo = snapshot_download('bodhan-ai/indic-ocr')
!cd {repo} && bash install.sh

Upload a zip of cleaned pages made locally with `autoassess ingest sheet.pdf pages/S001` (one folder per student).

In [ ]:
from google.colab import files
up = files.upload()
zip_name = next(iter(up))
!rm -rf pages && mkdir pages && unzip -q {zip_name} -d pages && find pages -name '*.png' | head

In [ ]:
import json, sys, pathlib
sys.path.insert(0, repo)
from indic_ocr import IndicOCR
parser = IndicOCR.from_pretrained(repo, table_format='markdown')

out = pathlib.Path('ocr_json')
for img in sorted(pathlib.Path('pages').rglob('*.png')):
    rel = img.relative_to('pages').with_suffix('.json')
    target = out / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    page = parser.parse(str(img))
    target.write_text(json.dumps(page, ensure_ascii=False, indent=2), encoding='utf-8')
    print(rel, len(page.get('blocks', [])), 'blocks')

In [ ]:
!zip -qr ocr_json.zip ocr_json
files.download('ocr_json.zip')

Locally: unzip into `ocr_json/`, then for each student

```bash
AUTOASSESS_OCR_PROVIDER=indic-precomputed AUTOASSESS_OCR_JSON_DIR=ocr_json/S001 \
  autoassess grade --spec exam_spec.yaml --out runs/indic pages/S001
```